In [8]:
# import necessary libraries needed for fine-tuning
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import pandas as pd
from datasets import Dataset
import evaluate
import numpy as np

# load the preprocessed local CSVs
train_df = pd.read_csv("../data/cleaned/finance/train.csv")
test_df = pd.read_csv("../data/cleaned/finance/test.csv")

train_df_clean = train_df.dropna(subset=["Content", "Summary"]) 
test_df_clean = test_df.dropna(subset=["Content", "Summary"])

# convert pandas dataframe into a hugging face dataset object
train_dataset = Dataset.from_pandas(train_df_clean.reset_index(drop=True)) # deleting the old index column entirely
test_dataset = Dataset.from_pandas(test_df_clean.reset_index(drop=True)) # deleting the old index column entirely

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("../models/bart-base", use_fast=True) # "use_fast" makes use of the faster tokenizerw 

# define tokenization function
def tokenize_function(dataset):
    # turns the input text into numerical tokens
    inputs = tokenizer(
        dataset["Content"],
        max_length=512,
        truncation=True,
        padding=False
    )

    # tokenize the labels (summary)
    with tokenizer.as_target_tokenizer(): 
        labels = tokenizer( 
            dataset["Summary"], 
            max_length=128, 
            truncation=True, 
            padding=False 
        )

    inputs["labels"] = labels["input_ids"]
    return inputs

# tokenizing the hugging face dataset object
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["Content", "Summary"]) # runs in a batch of 1000 by default
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["Content", "Summary"])

# Load the pre-trained BART model
model = AutoModelForSeq2SeqLM.from_pretrained("../models/bart-base/")
model.gradient_checkpointing_enable() # saves GPU VRAM by recomputing some activations on the fly during back propagation, to compute gradients.

# data collator
data_collation = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model) # helps in dynamic padding of rows in a single batch

# define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="../outputs/fine-tuned-bart/", # output directory for the train outputs
    evaluation_strategy="epoch", # tells the trainer when to run the evaluation strat
    save_strategy="epoch", # save the version after each epoch
    save_total_limit=2, # saves two versions
    load_best_model_at_end=True,
    metric_for_best_model="rougeL", 
    greater_is_better=True,

    learning_rate=3e-5, # controls how big of a step the optimizer takes when updating the model weights 
    per_device_train_batch_size=1, # number of samples processed in parallel per GPU during training
    per_device_eval_batch_size=1, # batch size for validation forward passes
    gradient_accumulation_steps=8, # Accumulate gradients for 8 mini-batches before updating weights -> simulates a larger effective batch size without extra GPU memory
    gradient_checkpointing=True, # Save GPU memory by recomputing certain activations during backpropagation instead of storing them
    fp16=False, # Uses mixed precision training (16-bit floats instead of 32-bit floats) if GPU supports it (NVIDIA GPU with Tensor Cores). Speeds up training.

    num_train_epochs=3, # number of full passes through the dataset
    weight_decay=0.01, # Discourages weights from growing too large to prevent overfitting
    predict_with_generate=True, # during evaluation, it makes the model generate summaries (instead of just computing loss)
    logging_dir="../outputs/fine-tuned-bart/logs/",
    logging_strategy="epoch" 
)

# load the ROUGE score metric
rouge = evaluate.load("rouge")
# bert = evaluate.load("bertscore")

# define compute_metrics() function
def compute_metrics(eval_pred):
    # generate predictions
    # 1. obtain the predictions and labels
    predictions, labels = eval_pred

    # 2. Replace -100 with pad_token_id for decoding tokens into text
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 3. decode to tokens into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 4. ROUGE requires newline 
    decoded_preds = ["\n".join(preds.strip().split(". ")) for preds in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split(". ")) for label in decoded_labels]

    # 5. compute ROUGE/BERTScore
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels,use_stemmer=True, use_aggregator=True)
    # bertscore_result = bert.compute(predictions=decoded_preds, references=decoded_labels, lang="en")

    # 6. create a single dictionary of metrics
    result = {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        # "bertscore_f1": np.mean(bertscore_result["f1"]),
    }

    # 7. return in percent
    return {k: round(v * 100, 4) for k, v in result.items()}

# for smoke test purposes
tiny_train = tokenized_train.select(range(64))
tiny_eval  = tokenized_test.select(range(32))

# config the trainer
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tiny_train,    
    eval_dataset = tiny_eval,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics,
    data_collator=data_collation,
)

# run the trainer
trainer.train()

# evaluate the trainer
eval_metrics = trainer.evaluate()
print("eval: ", eval_metrics)

# save the fine-tuned model + tokenizer
model.save_pretrained("../outputs/fine-tuned-bart")
tokenizer.save_pretrained("../outputs/fine-tuned-bart")


Map: 100%|██████████| 16623/16623 [00:04<00:00, 3645.99 examples/s]
d:\Programming\Project\News-Summarizer-LLM\venv\lib\site-packages\transformers\modeling_utils.py:367: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issu

{'loss': 0.5804, 'learning_rate': 1.9999999999999998e-05, 'epoch': 1.0}


                                              
 33%|███▎      | 8/24 [02:28<02:59, 11.24s/it] Saving model checkpoint to ../outputs/fine-tuned-bart/checkpoint-8
Configuration saved in ../outputs/fine-tuned-bart/checkpoint-8\config.json


{'eval_loss': 0.0010853323619812727, 'eval_rouge1': 79.4517, 'eval_rouge2': 77.9973, 'eval_rougeL': 79.3666, 'eval_runtime': 60.6455, 'eval_samples_per_second': 0.528, 'eval_steps_per_second': 0.528, 'epoch': 1.0}


Model weights saved in ../outputs/fine-tuned-bart/checkpoint-8\pytorch_model.bin
tokenizer config file saved in ../outputs/fine-tuned-bart/checkpoint-8\tokenizer_config.json
Special tokens file saved in ../outputs/fine-tuned-bart/checkpoint-8\special_tokens_map.json
d:\Programming\Project\News-Summarizer-LLM\venv\lib\site-packages\torch\_dynamo\eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
d:\Programming\Project\News-Summarizer-LLM\venv\lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.ena

...

{'train_runtime': 392.3669, 'train_samples_per_second': 0.489, 'train_steps_per_second': 0.061, 'train_loss': 0.478355477253596, 'epoch': 3.0}


100%|██████████| 32/32 [00:44<00:00,  1.40s/it]
Configuration saved in ../outputs/fine-tuned-bart\config.json


eval:  {'eval_loss': 0.0010853323619812727, 'eval_rouge1': 79.4517, 'eval_rouge2': 77.9973, 'eval_rougeL': 79.3666, 'eval_runtime': 46.2425, 'eval_samples_per_second': 0.692, 'eval_steps_per_second': 0.692, 'epoch': 3.0}


Model weights saved in ../outputs/fine-tuned-bart\pytorch_model.bin
tokenizer config file saved in ../outputs/fine-tuned-bart\tokenizer_config.json
Special tokens file saved in ../outputs/fine-tuned-bart\special_tokens_map.json


('../outputs/fine-tuned-bart\\tokenizer_config.json',
 '../outputs/fine-tuned-bart\\special_tokens_map.json',
 '../outputs/fine-tuned-bart\\vocab.json',
 '../outputs/fine-tuned-bart\\merges.txt',
 '../outputs/fine-tuned-bart\\added_tokens.json',
 '../outputs/fine-tuned-bart\\tokenizer.json')

In [7]:
import torch
print(torch.cuda.is_available())  # should be True
# print(torch.cuda.get_device_name(0))

False
